In [53]:
import pandas as pd
from pathlib import Path

BASE = Path.cwd().parents[1] / 'data' / 'raw'

# Cargar todas las bases
precio_bolsa = pd.read_parquet(BASE / 'precio_bolsa.parquet')
aportes = pd.read_parquet(BASE / 'aportes_energia.parquet')
vertimientos = pd.read_parquet(BASE / 'vertimientos.parquet')
generacion = pd.read_parquet(BASE / 'generacion_real.parquet')
disponibilidad = pd.read_parquet(BASE / 'disponibilidad_real.parquet')
demanda = pd.read_parquet(BASE / 'demanda_real.parquet')
escasez = pd.read_parquet(BASE / 'precio_escasez.parquet')
oni = pd.read_parquet(BASE / 'oni.parquet')
trm = pd.read_parquet(BASE / 'trm.parquet')

print("Bases cargadas:")
for nombre, df in [('precio_bolsa', precio_bolsa), ('aportes', aportes), 
                    ('vertimientos', vertimientos), ('generacion', generacion),
                    ('disponibilidad', disponibilidad), ('demanda', demanda),
                    ('escasez', escasez), ('oni', oni), ('trm', trm)]:
    print(f"  {nombre}: {df.shape} — {df.columns.tolist()}")

Bases cargadas:
  precio_bolsa: (4885, 6) — ['CodigoVariable', 'Fecha', 'CodigoDuracion', 'UnidadMedida', 'Version', 'Valor']
  aportes: (190257, 7) — ['FechaPublicacion', 'Fecha', 'CodigoSerieHidrologica', 'RegionHidrologica', 'AportesHidricosEnergia', 'PromedioAcumuladoEnergia', 'MediaHistoricaEnergia']
  vertimientos: (93558, 4) — ['Fecha', 'CodigoEmbalse', 'VertimientosEnergia', 'CodigoDuracion']
  generacion: (1261783, 8) — ['Fecha', 'CodigoPlanta', 'TipoGeneracion', 'TipoClasificacion', 'TipoDespachoRecurso', 'GeneracionRealEstimada', 'GeneracionProgramadaDespacho', 'GeneracionProgramadaRedespacho']
  disponibilidad: (850572, 6) — ['Fecha', 'CodigoUnidadGeneracion', 'TipoGeneracion', 'CapacidadEfectivaNeta', 'PromedioDisponibilidadReal', 'PorcentajePromedioDisponibilidadReal']
  demanda: (3957984, 8) — ['CodigoVariable', 'Fecha', 'CodigoSICAgente', 'TipoMercado', 'Version', 'Valor', 'UnidadMedida', 'CodigoDuracion']
  escasez: (525, 4) — ['CodigoVariable', 'Fecha', 'CodigoDuracio

In [54]:
bases = {
    'precio_bolsa': precio_bolsa,
    'aportes': aportes,
    'vertimientos': vertimientos,
    'generacion': generacion,
    'disponibilidad': disponibilidad,
    'demanda': demanda,
    'escasez': escasez,
    'oni': oni,
    'trm': trm
}

for nombre, df in bases.items():
    fecha = pd.to_datetime(df['Fecha'])
    print(f"{nombre}: {fecha.min().date()} → {fecha.max().date()}")

precio_bolsa: 2013-01-01 → 2026-08-17
aportes: 2013-01-01 → 2026-08-19
vertimientos: 2013-01-01 → 2026-08-19
generacion: 2013-01-01 → 2026-08-18
disponibilidad: 2013-01-01 → 2026-08-19
demanda: 2021-01-01 → 2026-08-15
escasez: 2013-01-01 → 2026-08-01
oni: 1950-01-01 → 2026-07-01
trm: 2013-01-03 → 2026-06-19


In [55]:
df_precio = precio_bolsa[['Fecha', 'Valor']].copy()
df_precio.columns = ['Fecha', 'precio_bolsa']
df_precio['Fecha'] = pd.to_datetime(df_precio['Fecha'])
df_precio = df_precio.sort_values('Fecha').reset_index(drop=True)
print(df_precio.head())
print(df_precio.shape)

       Fecha  precio_bolsa
0 2013-01-01      161.8653
1 2013-01-02      193.6470
2 2013-01-03      184.4642
3 2013-01-04      188.2384
4 2013-01-05      178.6095
(4885, 2)


In [56]:
df_aportes = aportes[['Fecha', 'AportesHidricosEnergia']].copy()
df_aportes['Fecha'] = pd.to_datetime(df_aportes['Fecha'])
df_aportes = df_aportes.groupby('Fecha')['AportesHidricosEnergia'].sum().reset_index()
df_aportes.columns = ['Fecha', 'aportes_energia_gwh']
df_aportes = df_aportes.sort_values('Fecha').reset_index(drop=True)
print(df_aportes.head())
print(df_aportes.shape)

       Fecha  aportes_energia_gwh
0 2013-01-01          124884800.0
1 2013-01-02          133792600.0
2 2013-01-03          122927200.0
3 2013-01-04          108007000.0
4 2013-01-05          111063800.0
(4979, 2)


In [57]:
df_vertimientos = vertimientos[['Fecha', 'VertimientosEnergia']].copy()
df_vertimientos['Fecha'] = pd.to_datetime(df_vertimientos['Fecha'])
df_vertimientos = df_vertimientos.groupby('Fecha')['VertimientosEnergia'].sum().reset_index()
df_vertimientos.columns = ['Fecha', 'vertimientos_energia_gwh']
df_vertimientos = df_vertimientos.sort_values('Fecha').reset_index(drop=True)
print(df_vertimientos.head())
print(df_vertimientos.shape)

       Fecha  vertimientos_energia_gwh
0 2013-01-01                      0.00
1 2013-01-02                 278217.60
2 2013-01-03                 215713.92
3 2013-01-04                      0.00
4 2013-01-05                      0.00
(3510, 2)


In [58]:
df_generacion = generacion[['Fecha', 'TipoGeneracion', 'GeneracionRealEstimada']].copy()
df_generacion['Fecha'] = pd.to_datetime(df_generacion['Fecha'])
df_generacion = df_generacion.groupby(['Fecha', 'TipoGeneracion'])['GeneracionRealEstimada'].sum().unstack(fill_value=0).reset_index()
df_generacion.columns.name = None
df_generacion.columns = ['Fecha'] + [f'gen_{c.lower().replace(" ", "_")}' for c in df_generacion.columns[1:]]
df_generacion = df_generacion.sort_values('Fecha').reset_index(drop=True)
print(df_generacion.head())
print(df_generacion.shape)
print(df_generacion.columns.tolist())

       Fecha  gen_cogenerador  gen_eolica  gen_hidraulica  gen_solar  \
0 2013-01-01        491244.73   182644.13    8.746490e+07        0.0   
1 2013-01-02        479816.69   178536.06    1.008231e+08        0.0   
2 2013-01-03        481465.91   185739.95    1.118072e+08        0.0   
3 2013-01-04        532811.47   207817.63    1.155799e+08        0.0   
4 2013-01-05        528232.27   238753.31    1.047845e+08        0.0   

   gen_termica  
0  37567779.87  
1  50853051.00  
2  48810850.48  
3  47282234.55  
4  51259272.59  
(4978, 6)
['Fecha', 'gen_cogenerador', 'gen_eolica', 'gen_hidraulica', 'gen_solar', 'gen_termica']


In [59]:
df_disponibilidad = disponibilidad[['Fecha', 'TipoGeneracion', 'PorcentajePromedioDisponibilidadReal']].copy()
df_disponibilidad['Fecha'] = pd.to_datetime(df_disponibilidad['Fecha'])
df_disponibilidad = df_disponibilidad.groupby(['Fecha', 'TipoGeneracion'])['PorcentajePromedioDisponibilidadReal'].mean().unstack(fill_value=0).reset_index()
df_disponibilidad.columns.name = None
df_disponibilidad.columns = ['Fecha'] + [f'disp_{c.lower().replace(" ", "_")}' for c in df_disponibilidad.columns[1:]]
df_disponibilidad = df_disponibilidad.sort_values('Fecha').reset_index(drop=True)
print(df_disponibilidad.head())
print(df_disponibilidad.columns.tolist())
print(df_disponibilidad.shape)

       Fecha  disp_cogenerador  disp_hidraulica  disp_solar  disp_termica
0 2013-01-01               0.0         0.913226         0.0      0.946047
1 2013-01-02               0.0         0.911720         0.0      0.950000
2 2013-01-03               0.0         0.912473         0.0      0.937442
3 2013-01-04               0.0         0.901828         0.0      0.921860
4 2013-01-05               0.0         0.911613         0.0      0.909302
['Fecha', 'disp_cogenerador', 'disp_hidraulica', 'disp_solar', 'disp_termica']
(4976, 5)


In [60]:
df_demanda = demanda[['Fecha', 'Valor']].copy()
df_demanda['Fecha'] = pd.to_datetime(df_demanda['Fecha']).dt.normalize()
df_demanda = df_demanda.groupby('Fecha')['Valor'].sum().reset_index()
df_demanda.columns = ['Fecha', 'demanda_total_kwh']
df_demanda = df_demanda.sort_values('Fecha').reset_index(drop=True)
print(df_demanda.head())
print(df_demanda.shape)

       Fecha  demanda_total_kwh
0 2021-01-01       1.522788e+08
1 2021-01-02       1.650900e+08
2 2021-01-03       1.649372e+08
3 2021-01-04       1.864151e+08
4 2021-01-05       1.898693e+08
(1836, 2)


In [61]:
print(escasez.groupby(['Fecha', 'CodigoVariable']).size().reset_index(name='count').query('count > 1'))

          Fecha           CodigoVariable  count
218  2022-05-01            PrecioEscasez      2
219  2022-05-01  PrecioEscasezActivacion      2
220  2022-05-01    PrecioMarginalEscasez      2
221  2022-06-01            PrecioEscasez      3
222  2022-06-01  PrecioEscasezActivacion      3
..          ...                      ...    ...
365  2026-05-01    PrecioEscasezSuperior      2
366  2026-06-01            PrecioEscasez      3
367  2026-06-01  PrecioEscasezActivacion      3
368  2026-06-01    PrecioEscasezInferior      3
369  2026-06-01    PrecioEscasezSuperior      3

[140 rows x 3 columns]


In [62]:
print(escasez['CodigoVariable'].unique())

['PrecioEscasez' 'PrecioEscasezActivacion' 'PrecioMarginalEscasez'
 'PrecioEscasezInferior' 'PrecioEscasezSuperior']


In [63]:
print(escasez[escasez['Fecha'] == '2022-05-01'].sort_values('CodigoVariable'))

              CodigoVariable       Fecha CodigoDuracion      Valor
221            PrecioEscasez  2022-05-01            P1M   971.1801
222            PrecioEscasez  2022-05-01            P1M   971.1801
219  PrecioEscasezActivacion  2022-05-01            P1M  1100.2206
223  PrecioEscasezActivacion  2022-05-01            P1M  1100.2206
218    PrecioMarginalEscasez  2022-05-01            P1M  1100.2206
220    PrecioMarginalEscasez  2022-05-01            P1M  1100.2206


In [64]:
df_escasez = escasez[escasez['CodigoVariable'] == 'PrecioEscasez'][['Fecha', 'Valor']].copy()
df_escasez['Fecha'] = pd.to_datetime(df_escasez['Fecha'])
df_escasez = df_escasez.drop_duplicates('Fecha')
df_escasez.columns = ['Fecha', 'precio_escasez']
df_escasez = df_escasez.sort_values('Fecha').reset_index(drop=True)
print(df_escasez.head())
print(df_escasez.shape)

       Fecha  precio_escasez
0 2013-01-01          430.21
1 2013-02-01          423.48
2 2013-03-01          447.55
3 2013-04-01          462.52
4 2013-05-01          445.80
(164, 2)


In [65]:
df_oni = oni.copy()
df_oni['Fecha'] = pd.to_datetime(df_oni['Fecha'])
df_oni = df_oni.sort_values('Fecha').reset_index(drop=True)

df_trm = trm.copy()
df_trm['Fecha'] = pd.to_datetime(df_trm['Fecha'])
df_trm = df_trm.sort_values('Fecha').reset_index(drop=True)

print(df_oni.head())
print(df_trm.head())

       Fecha   ONI
0 1950-01-01 -1.19
1 1950-02-01 -1.08
2 1950-03-01 -0.96
3 1950-04-01 -1.00
4 1950-05-01 -0.99
       Fecha      TRM
0 2013-01-03  1759.97
1 2013-01-04  1760.83
2 2013-01-05  1767.54
3 2013-01-09  1771.31
4 2013-01-10  1767.96


In [66]:
# Crear índice de fechas diarias
fecha_inicio = '2013-01-01'
fecha_fin = '2026-07-31'
fechas = pd.DataFrame({'Fecha': pd.date_range(fecha_inicio, fecha_fin, freq='D')})

# Merge con todas las variables
df = fechas.copy()
df = df.merge(df_precio, on='Fecha', how='left')
df = df.merge(df_aportes, on='Fecha', how='left')
df = df.merge(df_vertimientos, on='Fecha', how='left')
df = df.merge(df_generacion, on='Fecha', how='left')
df = df.merge(df_disponibilidad, on='Fecha', how='left')
df = df.merge(df_escasez, on='Fecha', how='left')
df = df.merge(df_oni, on='Fecha', how='left')
df = df.merge(df_trm, on='Fecha', how='left')

# Forward fill para variables mensuales
df[['precio_escasez', 'ONI']] = df[['precio_escasez', 'ONI']].ffill()

# TRM forward fill para fines de semana y festivos
df['TRM'] = df['TRM'].ffill()

# Vertimientos: NaN significa 0
df['vertimientos_energia_gwh'] = df['vertimientos_energia_gwh'].fillna(0)

print(df.shape)
print(df.isnull().sum())

(4960, 16)
Fecha                        0
precio_bolsa                92
aportes_energia_gwh          0
vertimientos_energia_gwh     0
gen_cogenerador              0
gen_eolica                   0
gen_hidraulica               0
gen_solar                    0
gen_termica                  0
disp_cogenerador             3
disp_hidraulica              3
disp_solar                   3
disp_termica                 3
precio_escasez               0
ONI                          0
TRM                          2
dtype: int64


In [67]:
nulos = df[df['precio_bolsa'].isnull()]['Fecha']
print(f"Primer nulo: {nulos.min()}")
print(f"Último nulo: {nulos.max()}")
print(f"Total días: {len(nulos)}")

Primer nulo: 2016-04-01 00:00:00
Último nulo: 2019-01-31 00:00:00
Total días: 92


In [68]:
df_nulos = df[df['precio_bolsa'].isnull()][['Fecha']].copy()
df_nulos['dia_semana'] = df_nulos['Fecha'].dt.day_name()
print(df_nulos['dia_semana'].value_counts())

dia_semana
Tuesday      14
Friday       13
Saturday     13
Sunday       13
Monday       13
Wednesday    13
Thursday     13
Name: count, dtype: int64


In [69]:
df_nulos['año'] = df_nulos['Fecha'].dt.year
print(df_nulos['año'].value_counts().sort_index())

año
2016    61
2019    31
Name: count, dtype: int64


In [70]:
fechas_nulas = df[df['precio_bolsa'].isnull()]['Fecha'].tolist()
precio_raw = pd.read_parquet(BASE / 'precio_bolsa.parquet')
precio_raw['Fecha'] = pd.to_datetime(precio_raw['Fecha'])
print(precio_raw[precio_raw['Fecha'].isin(fechas_nulas)][['Fecha', 'Version', 'Valor']].head(20))

Empty DataFrame
Columns: [Fecha, Version, Valor]
Index: []


In [71]:
bolsa_sinergox = pd.read_excel(BASE.parent / 'external' / 'bolsa_sinergox.xlsx')
bolsa_sinergox['Fecha'] = pd.to_datetime(bolsa_sinergox['Fecha'])
bolsa_sinergox['Precio_Bolsa'] = pd.to_numeric(bolsa_sinergox['Precio_Bolsa'], errors='coerce')
bolsa_sinergox.columns = ['Fecha', 'precio_bolsa']

# Ver cuántos de los días faltantes están en el Excel
fechas_nulas = df[df['precio_bolsa'].isnull()]['Fecha'].tolist()
encontrados = bolsa_sinergox[bolsa_sinergox['Fecha'].isin(fechas_nulas)]
print(f"Días faltantes: {len(fechas_nulas)}")
print(f"Encontrados en Sinergox: {len(encontrados)}")
print(encontrados.head())

Días faltantes: 92
Encontrados en Sinergox: 92
        Fecha  precio_bolsa
91 2016-04-01    804.580549
92 2016-04-02    745.201266
93 2016-04-03    610.887017
94 2016-04-04    684.596914
95 2016-04-05    632.040620


In [72]:
# Rellenar los días faltantes con los datos de Sinergox
df = df.set_index('Fecha')
encontrados = encontrados.set_index('Fecha')
df.loc[df['precio_bolsa'].isnull(), 'precio_bolsa'] = encontrados['precio_bolsa']
df = df.reset_index()

print(df['precio_bolsa'].isnull().sum())

0


In [73]:
# TRM forward fill
df['TRM'] = df['TRM'].ffill()

# Disponibilidad forward fill
cols_disp = ['disp_cogenerador', 'disp_hidraulica', 'disp_solar', 'disp_termica']
df[cols_disp] = df[cols_disp].ffill()

print(df.isnull().sum())

Fecha                       0
precio_bolsa                0
aportes_energia_gwh         0
vertimientos_energia_gwh    0
gen_cogenerador             0
gen_eolica                  0
gen_hidraulica              0
gen_solar                   0
gen_termica                 0
disp_cogenerador            0
disp_hidraulica             0
disp_solar                  0
disp_termica                0
precio_escasez              0
ONI                         0
TRM                         2
dtype: int64


In [74]:
df['TRM'] = df['TRM'].bfill()
print(df['TRM'].isnull().sum())

0


In [75]:
from pathlib import Path

PROCESSED = Path.cwd().parents[1] / 'data' / 'processed'
PROCESSED.mkdir(exist_ok=True)

df.to_parquet(PROCESSED / 'dataset_consolidado.parquet', index=False)
print(f"Dataset guardado: {df.shape}")
print(df.head())

Dataset guardado: (4960, 16)
       Fecha  precio_bolsa  aportes_energia_gwh  vertimientos_energia_gwh  \
0 2013-01-01      161.8653          124884800.0                      0.00   
1 2013-01-02      193.6470          133792600.0                 278217.60   
2 2013-01-03      184.4642          122927200.0                 215713.92   
3 2013-01-04      188.2384          108007000.0                      0.00   
4 2013-01-05      178.6095          111063800.0                      0.00   

   gen_cogenerador  gen_eolica  gen_hidraulica  gen_solar  gen_termica  \
0        491244.73   182644.13    8.746490e+07        0.0  37567779.87   
1        479816.69   178536.06    1.008231e+08        0.0  50853051.00   
2        481465.91   185739.95    1.118072e+08        0.0  48810850.48   
3        532811.47   207817.63    1.155799e+08        0.0  47282234.55   
4        528232.27   238753.31    1.047845e+08        0.0  51259272.59   

   disp_cogenerador  disp_hidraulica  disp_solar  disp_termica 

In [76]:
print(f"Shape: {df.shape}")
print(f"Columnas: {df.columns.tolist()}")
print(df.head(2))

Shape: (4960, 16)
Columnas: ['Fecha', 'precio_bolsa', 'aportes_energia_gwh', 'vertimientos_energia_gwh', 'gen_cogenerador', 'gen_eolica', 'gen_hidraulica', 'gen_solar', 'gen_termica', 'disp_cogenerador', 'disp_hidraulica', 'disp_solar', 'disp_termica', 'precio_escasez', 'ONI', 'TRM']
       Fecha  precio_bolsa  aportes_energia_gwh  vertimientos_energia_gwh  \
0 2013-01-01      161.8653          124884800.0                       0.0   
1 2013-01-02      193.6470          133792600.0                  278217.6   

   gen_cogenerador  gen_eolica  gen_hidraulica  gen_solar  gen_termica  \
0        491244.73   182644.13    8.746490e+07        0.0  37567779.87   
1        479816.69   178536.06    1.008231e+08        0.0  50853051.00   

   disp_cogenerador  disp_hidraulica  disp_solar  disp_termica  \
0               0.0         0.913226         0.0      0.946047   
1               0.0         0.911720         0.0      0.950000   

   precio_escasez   ONI      TRM  
0          430.21 -0.55  1

In [77]:
print(df['Fecha'].min(), '→', df['Fecha'].max())

2013-01-01 00:00:00 → 2026-07-31 00:00:00


In [78]:
import pandas as pd
from pathlib import Path

BASE = Path.cwd().parents[1] / 'data' / 'raw'
oni = pd.read_parquet(BASE / 'oni.parquet')
print(oni.tail(5))

         Fecha   ONI
914 2026-03-01 -0.44
915 2026-04-01 -0.04
916 2026-05-01  0.49
917 2026-06-01  0.98
918 2026-07-01  0.98


In [79]:
print(df.shape)
print(df['Fecha'].max())

(4960, 16)
2026-07-31 00:00:00


In [80]:
PROCESSED = Path.cwd().parents[1] / 'data' / 'processed'
df.to_parquet(PROCESSED / 'dataset_consolidado.parquet', index=False)
print("Dataset guardado")

Dataset guardado
